# Serving benchmark: qwen3-4b-explain

Transformers fp16 / NF4 baselines; GGUF f16 / Q8_0 / Q4_K_M on llama.cpp CUDA (T4) and Q8_0 / Q4_K_M on llama.cpp CPU (4 threads). Needs GPU T4 x2, Internet, secret `hg`.

In [ ]:
%pip install -q "bitsandbytes" "datasets" "distill @ git+https://github.com/MSR327/explain-distill.git@ecff07fa4c4ca38b3f0dc0c827b56c047ff5344e"
%pip uninstall -y -q torchao
COMMIT, NAME, SIZE = "ecff07fa4c4ca38b3f0dc0c827b56c047ff5344e", "qwen3-4b-explain", "4B"
GPU_N, LAT, CPU_N = 120, 20, 5

In [ ]:
import os, shutil, subprocess, sys, json, glob

from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hg")
api = HfApi()
USER = api.whoami()["name"]
DATA, MODEL_REPO = f"{USER}/explain-distill-data", f"{USER}/{NAME}"
os.makedirs("data", exist_ok=True)
shutil.copy(hf_hub_download(DATA, "pairs.jsonl", repo_type="dataset"), "data/pairs.jsonl")
MERGED = snapshot_download(MODEL_REPO, allow_patterns=["merged/*"]) + "/merged"
print("ready", MERGED, os.listdir(MERGED))


def sh(cmd, marker):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    lines = [x for x in p.stdout.splitlines() if x.startswith(marker)]
    print(*lines, sep="\n")
    if p.returncode or not lines:
        print("FAILED:", cmd, p.stderr[-3000:])
        return None
    return lines


def upload(system):
    for f in glob.glob(f"outputs/{system}*"):
        api.upload_file(path_or_fileobj=f, path_in_repo=f"bench/{os.path.basename(f)}", repo_id=DATA,
                        repo_type="dataset", commit_message=f"bench {system} (code {COMMIT})")

In [ ]:
# llama.cpp CUDA for compute capability 7.5: cached binaries if present, else build (~25 min) + cache.
import os, re, subprocess, tarfile
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import EntryNotFoundError

def _build_sh(cmd, log):
    import time
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
        t0 = time.time()
        while p.poll() is None:  # progress every minute: a silent 30-min build looks the same as a hung one
            time.sleep(60)
            pct = re.findall(r"\[ *(\d+)%\]", open(log).read()[-4000:])
            print(f"  {log}: {(time.time() - t0) / 60:.0f} min" + (f", {pct[-1]}%" if pct else ""), flush=True)
    if p.returncode:
        print(open(log).read()[-6000:])
        raise RuntimeError(f"failed: {cmd}")

_build_sh("(nvcc --version || /usr/local/cuda/bin/nvcc --version) 2>&1 || true", "nvcc.log")
cuda = (re.findall(r"release ([0-9.]+)", open("nvcc.log").read()) or ["unknown"])[0]
TARGETS = "llama-server llama-quantize".split()
_api = HfApi()
_repo = f"{_api.whoami()['name']}/explain-distill-data"
_tar = f"llama-cpp-bin/b54752df40-cuda{cuda}-sm75.tar.gz"
print("CUDA", cuda, "cache", _tar)
if not os.path.exists("llama.cpp/.git"):
    _build_sh("git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp "
       "&& git fetch -q --depth 1 origin b54752df40e4d8e53d34ef5a598336b4e4871d39 && git checkout -q FETCH_HEAD", "clone.log")
try:
    with tarfile.open(hf_hub_download(_repo, _tar, repo_type="dataset")) as t:
        t.extractall("llama.cpp", filter="data")
    print("using cached llama.cpp binaries")
except EntryNotFoundError:
    _build_sh("cd llama.cpp && cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
       "-DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release", "cmake.log")
    _build_sh("cd llama.cpp && cmake --build build -j 4 --target llama-server llama-quantize", "build.log")
    with tarfile.open("llama-bin.tar.gz", "w:gz") as t:
        t.add("llama.cpp/build/bin", arcname="build/bin")
    _api.upload_file(path_or_fileobj="llama-bin.tar.gz", path_in_repo=_tar, repo_id=_repo, repo_type="dataset",
                     commit_message=f"llama.cpp b54752df40 CUDA {cuda} sm_75 binaries")
    print("built and cached llama.cpp binaries")
for t in TARGETS:
    assert os.access(f"llama.cpp/build/bin/{t}", os.X_OK), t
print("llama.cpp ready:", *TARGETS)

In [ ]:
# Transformers baselines, one request at a time on GPU 0.
for tag, extra in (("hf-fp16", ""), ("hf-nf4", "--load-4bit")):
    sh(f"CUDA_VISIBLE_DEVICES=0 {sys.executable} -m distill.bench_hf --model {MERGED} --n {LAT} {extra} "
       f"--system {NAME}-{tag} --out outputs/{NAME}-{tag}.jsonl", "BENCH")
    upload(f"{NAME}-{tag}")

In [ ]:
# convert_hf_to_gguf.py imports the gguf-py folder next to it; pip-installing gguf-py would upgrade
# numpy (it needs >= 2.2.6) under the running session, so it is deliberately not installed (checked locally).
sh(f"{sys.executable} -m distill.llamacpp convert --merged {MERGED} --llama-cpp llama.cpp --out gguf", "GGUF")
print(*[f"{f} {os.path.getsize(f) / 1e9:.2f} GB" for f in sorted(glob.glob("gguf/*.gguf"))], sep="\n")

In [ ]:
# llama.cpp CUDA, one T4: 30 sequential requests (latency), then all eval prompts on 8 slots.
for q in ("f16", "Q8_0", "Q4_K_M"):
    system = f"{NAME}-llamacpp-cuda-{q}"
    sh(f"{sys.executable} -m distill.llamacpp bench --gguf gguf/model-{q}.gguf --llama-cpp llama.cpp "
       f"--tokenizer {MERGED} --gpu 0 --ngl 99 --parallel 8 --n {GPU_N} --n-latency {LAT} --system {system} "
       f"--out outputs/{system}.jsonl", "BENCH")
    upload(system)

In [ ]:
# llama.cpp CPU only (GPUs hidden: a CUDA build offloads prompt matmuls), 4 threads, 20 requests.
for q in ("Q8_0", "Q4_K_M"):
    system = f"{NAME}-llamacpp-cpu-{q}"
    sh(f"{sys.executable} -m distill.llamacpp bench --gguf gguf/model-{q}.gguf --llama-cpp llama.cpp "
       f"--tokenizer {MERGED} --gpu none --ngl 0 --threads 4 --n {CPU_N} --system {system} "
       f"--out outputs/{system}.jsonl", "BENCH")
    upload(system)
for q in ("Q8_0", "Q4_K_M"):
    api.upload_file(path_or_fileobj=f"gguf/model-{q}.gguf", path_in_repo=f"gguf/{NAME}-{q}.gguf",
                    repo_id=MODEL_REPO, commit_message=f"GGUF {q} (code {COMMIT})")
print("done")